# 3. Recopilación de Datos mediante Crowdsourcing

Recopilar datos de tráfico reportados por usuarios (conductores).

## Objetivo
- Generar reportes simulados de usuarios
- Validar ubicaciones GPS
- Detectar reportes fraudulentos
- Agregar por avenida y hora
- Exportar a `data/raw/`

In [1]:
# ============================================================================
# IMPORTAR LIBRERÍAS
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Configuración
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('default')
sns.set_palette('husl')

DATA_RAW = Path('../data/raw')

print('✓ Librerías cargadas')

✓ Librerías cargadas


## Configuración

In [2]:
# ============================================================================
# CONFIGURACIÓN
# ============================================================================
np.random.seed(42)

# Coordenadas de avenidas principales (lat, lon)
ROADS_COORDS = {
    'Av. Chapultepec': (20.6595, -103.3855),
    'Av. México': (20.6520, -103.3950),
    'Av. Vallarta': (20.6450, -103.4050),
    'Av. Gigantes': (20.6580, -103.4150),
    'Av. López Mateos': (20.6700, -103.3750)
}

# Límites de Guadalajara
GDL_BOUNDS = {
    'lat_min': 20.5,
    'lat_max': 20.8,
    'lon_min': -103.5,
    'lon_max': -103.2
}

print(f'✓ {len(ROADS_COORDS)} avenidas configuradas')

✓ 5 avenidas configuradas


## Funciones de Validación

In [3]:
# ============================================================================
# FUNCIONES
# ============================================================================
def validate_gps(lat, lon, bounds):
    """
    Validar que GPS esté dentro de Guadalajara.
    """
    return (bounds['lat_min'] <= lat <= bounds['lat_max'] and
            bounds['lon_min'] <= lon <= bounds['lon_max'])

def validate_speed(speed, min_speed=0, max_speed=150):
    """
    Validar que velocidad sea realista.
    """
    return min_speed < speed < max_speed

def generate_crowdsourcing_reports(num_reports=500):
    """
    Generar reportes simulados de usuarios.
    """
    records = []
    
    for i in range(num_reports):
        road = np.random.choice(list(ROADS_COORDS.keys()))
        lat, lon = ROADS_COORDS[road]
        
        # Agregar ruido GPS realista
        lat += np.random.normal(0, 0.001)
        lon += np.random.normal(0, 0.001)
        
        timestamp = datetime.now() - timedelta(days=np.random.randint(0, 30))
        
        records.append({
            'user_id': f'user_{np.random.randint(1000, 9999)}',
            'timestamp': timestamp,
            'road': road,
            'latitude': round(lat, 6),
            'longitude': round(lon, 6),
            'speed_kmh': np.random.normal(45, 15),
            'report_type': np.random.choice(['congestion', 'incident', 'roadwork', 'normal']),
            'confidence': np.random.uniform(0.5, 1.0),
            'device_type': np.random.choice(['mobile', 'web']),
            'app_version': np.random.choice(['1.0', '1.1', '1.2'])
        })
    
    return pd.DataFrame(records)

## Generar Reportes

In [4]:
# ============================================================================
# GENERAR REPORTES
# ============================================================================
print('Generando reportes de crowdsourcing...')
df_raw = generate_crowdsourcing_reports(num_reports=500)
print(f'✓ {len(df_raw)} reportes generados')
print(f'\nPrimeros registros:')
print(df_raw.head())

Generando reportes de crowdsourcing...
✓ 500 reportes generados

Primeros registros:
     user_id                  timestamp              road   latitude  \
0  user_5426 2026-04-09 22:03:08.043721      Av. Gigantes  20.656888   
1  user_6311 2026-04-26 22:03:08.043974        Av. México  20.651507   
2  user_3558 2026-04-01 22:03:08.044092  Av. López Mateos  20.668275   
3  user_4556 2026-04-03 22:03:08.044246      Av. Vallarta  20.644258   
4  user_8041 2026-04-07 22:03:08.044511        Av. México  20.651400   

    longitude  speed_kmh report_type  confidence device_type app_version  
0 -103.414681  14.835557      normal    0.854036         web         1.0  
1 -103.394767  46.769919      normal    0.591702         web         1.1  
2 -103.375562  38.579309      normal    0.616386         web         1.2  
3 -103.404858  44.480217    roadwork    0.840154      mobile         1.2  
4 -103.394053  35.974401    roadwork    0.804998         web         1.2  


## Validar Datos

In [5]:
# ============================================================================
# VALIDAR
# ============================================================================
print('\nValidando datos...')

# Validar GPS
df_raw['valid_gps'] = df_raw.apply(
    lambda row: validate_gps(row['latitude'], row['longitude'], GDL_BOUNDS),
    axis=1
)

# Validar velocidad
df_raw['valid_speed'] = df_raw['speed_kmh'].apply(validate_speed)

# Resumen de validación
gps_invalid = (~df_raw['valid_gps']).sum()
speed_invalid = (~df_raw['valid_speed']).sum()

print(f'  GPS inválido: {gps_invalid} reportes')
print(f'  Velocidad inválida: {speed_invalid} reportes')

# Filtrar reportes válidos
df_clean = df_raw[df_raw['valid_gps'] & df_raw['valid_speed']].copy()
df_clean = df_clean.drop(['valid_gps', 'valid_speed'], axis=1)

print(f'\n✓ Reportes válidos: {len(df_clean)} de {len(df_raw)} ({len(df_clean)/len(df_raw)*100:.1f}%)')


Validando datos...
  GPS inválido: 0 reportes
  Velocidad inválida: 0 reportes

✓ Reportes válidos: 500 de 500 (100.0%)


## Agregar Datos

In [8]:
# ============================================================================
# AGREGAR POR AVENIDA Y HORA
# ============================================================================
print('\nAgregando por avenida y hora...')

# Agrupar por hora y avenida
df_clean['hour'] = df_clean['timestamp'].dt.floor('h')

aggregated = df_clean.groupby(['hour', 'road']).agg({
    'speed_kmh': ['mean', 'std', 'min', 'max'],
    'confidence': 'mean',
    'report_type': lambda x: (x == 'congestion').sum(),
    'user_id': 'nunique'
}).reset_index()

# Renombrar columnas
aggregated.columns = [
    'timestamp', 'road',
    'avg_speed', 'std_speed', 'min_speed', 'max_speed',
    'avg_confidence', 'congestion_reports', 'unique_users'
]

print(f'✓ {len(aggregated)} registros agregados')
print(f'\nEjemplos:')
print(aggregated.head())


Agregando por avenida y hora...
✓ 148 registros agregados

Ejemplos:
            timestamp              road  avg_speed  std_speed  min_speed  \
0 2026-03-29 22:00:00   Av. Chapultepec  61.742689  15.947994  45.587459   
1 2026-03-29 22:00:00      Av. Gigantes  46.969871   7.380987  38.225108   
2 2026-03-29 22:00:00  Av. López Mateos  57.870176   6.839784  47.270542   
3 2026-03-29 22:00:00        Av. México  46.641620  13.940932  30.241007   
4 2026-03-29 22:00:00      Av. Vallarta  49.224169  13.893318  36.393693   

   max_speed  avg_confidence  congestion_reports  unique_users  
0  77.475035        0.670989                   0             3  
1  55.157791        0.705202                   0             5  
2  65.037085        0.836661                   0             5  
3  63.088728        0.769968                   0             4  
4  61.291583        0.756640                   0             4  


## Exportar

In [9]:
# ============================================================================
# EXPORTAR
# ============================================================================
DATA_RAW.mkdir(parents=True, exist_ok=True)

# Exportar datos sin procesar (validados)
raw_file = DATA_RAW / 'crowdsourcing_raw.csv'
df_clean.to_csv(raw_file, index=False)

# Exportar datos agregados
agg_file = DATA_RAW / 'crowdsourcing_aggregated.csv'
aggregated.to_csv(agg_file, index=False)

print('✓ Exportación completada:')
print(f'  Raw: {raw_file}')
print(f'  Agregado: {agg_file}')

✓ Exportación completada:
  Raw: ..\data\raw\crowdsourcing_raw.csv
  Agregado: ..\data\raw\crowdsourcing_aggregated.csv
